# 🎯 Hazeon AI — Fine-tune Llama-3 on UPSC Topper Answers

**What this notebook does:**
1. Loads your pre-built 908-example dataset (or scrapes fresh data)
2. Fine-tunes Llama-3.1-8B with anti-hallucination training
3. Saves the model to Google Drive (persists after Colab disconnects)

**Before running:**
1. `Runtime → Change runtime type → T4 GPU`
2. Upload `upsc_eval_dataset.jsonl` to your Google Drive root folder

**Total runtime:** ~1.5 hours on T4 (if dataset pre-loaded from Drive)

In [ ]:
# Step 0: Mount Google Drive — model will be saved here permanently
from google.colab import drive
drive.mount('/content/drive')

import os

DRIVE_BASE = '/content/drive/MyDrive/hazeon_finetune'
os.makedirs(DRIVE_BASE, exist_ok=True)
os.makedirs(f'{DRIVE_BASE}/training_data', exist_ok=True)

DATASET_PATH = f'{DRIVE_BASE}/training_data/upsc_eval_dataset.jsonl'
MODEL_OUTPUT_PATH = f'{DRIVE_BASE}/hazeon-upsc-evaluator'

# Check if pre-built dataset exists in Drive
if os.path.exists(DATASET_PATH):
    line_count = sum(1 for _ in open(DATASET_PATH))
    print(f'✅ Found pre-built dataset in Drive: {line_count} training examples')
    print('   Will SKIP scraping step and load directly.')
    USE_PREBUILT_DATASET = True
else:
    print('⚠️  Dataset not found in Drive.')
    print(f'   Expected: {DATASET_PATH}')
    print('   Upload upsc_eval_dataset.jsonl to MyDrive/hazeon_finetune/training_data/')
    print('   OR set USE_PREBUILT_DATASET = False to build from scraping (takes 2-3hrs)')
    USE_PREBUILT_DATASET = False

print(f'\nDrive folder: {DRIVE_BASE}')
print(f'Model will be saved to: {MODEL_OUTPUT_PATH}')

In [ ]:
# Step 1: Install all dependencies
!pip install -q unsloth transformers datasets trl torch accelerate bitsandbytes
!pip install -q beautifulsoup4 lxml httpx chromadb
print('✅ Dependencies installed')

In [ ]:
# Step 2: Verify GPU
import torch
print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    raise RuntimeError('❌ No GPU — go to Runtime → Change runtime type → T4 GPU')

In [ ]:
# Step 3: Load pre-built dataset OR scrape from web
import json

training_examples = []

if USE_PREBUILT_DATASET:
    # ── Fast path: load the 908-example dataset you built locally ──
    with open(DATASET_PATH, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                training_examples.append(json.loads(line))
    print(f'✅ Loaded {len(training_examples)} training examples from Drive')
    tier_counts = {}
    for ex in training_examples:
        out = json.loads(ex['output']) if isinstance(ex['output'], str) else ex['output']
        fb = out.get('feedback_summary', '')
        if 'excellent' in fb.lower() or 'near topper' in fb.lower(): tier = 'good'
        elif 'covers basics' in fb.lower(): tier = 'average'
        elif 'very basic' in fb.lower(): tier = 'very_weak'
        else: tier = 'other'
        tier_counts[tier] = tier_counts.get(tier, 0) + 1
    print(f'   Tier breakdown: {tier_counts}')
else:
    print('Running scraping pipeline — this will take 2-3 hours...')
    print('Tip: Upload upsc_eval_dataset.jsonl to Drive to skip this next time!')

In [ ]:
# Step 3b: SCRAPING (only runs if USE_PREBUILT_DATASET = False)
# Skip this cell if you loaded the pre-built dataset above

if not USE_PREBUILT_DATASET:
    import re, time, logging
    from dataclasses import dataclass, field
    from typing import List, Optional

    logging.basicConfig(level=logging.INFO)
    logger = logging.getLogger('scraper')

    @dataclass
    class TopperRecord:
        question: str
        answer: str
        subject: str
        year: int
        source: str
        score: float = 8.5
        rank: Optional[int] = None

    def fetch(url, timeout=15):
        try:
            import httpx
            r = httpx.get(url, headers={'User-Agent': 'Mozilla/5.0'}, timeout=timeout, follow_redirects=True)
            return r.text if r.status_code == 200 else None
        except: return None

    def clean(html):
        html = re.sub(r'<[^>]+>', ' ', html)
        html = re.sub(r'&nbsp;|&amp;|&lt;|&gt;', ' ', html)
        return re.sub(r'\s+', ' ', html).strip()

    def extract_body(html):
        try:
            from bs4 import BeautifulSoup
            soup = BeautifulSoup(html, 'html.parser')
            for tag in soup(['nav','footer','aside','script','style','header']):
                tag.decompose()
            for sel in ['article .entry-content','article','.post-content','.entry-content','main']:
                el = soup.select_one(sel)
                if el: return clean(el.get_text(' '))
            return clean(soup.get_text(' '))
        except: return clean(html)

    def extract_qa_pairs(text):
        pairs = []
        q_re = re.compile(r'(?:Q\.?\s*\d*\.?|Question\s*\d*\.?)\s*(.+?)(?=\n\s*(?:Ans|Answer|A\.))', re.I|re.DOTALL)
        a_re = re.compile(r'(?:Ans|Answer|A\.)\s*[:.]?\s*(.+?)(?=\n\s*(?:Q\.?\s*\d+|Question\s*\d+)|$)', re.I|re.DOTALL)
        for q, a in zip(q_re.findall(text), a_re.findall(text)):
            if len(q.strip()) > 20 and len(a.strip()) > 50:
                pairs.append({'q': q.strip()[:500], 'a': a.strip()[:3000]})
        return pairs

    def infer_subject(text):
        t = text.lower()
        if any(w in t for w in ['ethics','integrity','moral','values']): return 'GS4 - Ethics'
        if any(w in t for w in ['governance','constitution','parliament','federalism','foreign policy']): return 'GS2 - Governance & IR'
        if any(w in t for w in ['economy','agriculture','environment','technology','security']): return 'GS3 - Economy & Environment'
        if any(w in t for w in ['history','culture','society','geography','gender']): return 'GS1 - History & Society'
        return 'GS - General Studies'

    all_records = []

    SOURCES = [
        ('InsightsIAS', [
            'https://www.insightsonindia.com/category/upsc-mains-model-answers/gs-1-model-answers/',
            'https://www.insightsonindia.com/category/upsc-mains-model-answers/gs-2-model-answers/',
            'https://www.insightsonindia.com/category/upsc-mains-model-answers/gs-3-model-answers/',
            'https://www.insightsonindia.com/category/upsc-mains-model-answers/gs-4-model-answers/',
        ], 15),
        ('CivilsDaily', [
            'https://www.civilsdaily.com/upsc-mains-2024-gs-model-solutions-gs1-gs2-gs3-gs4/',
            'https://www.civilsdaily.com/upsc-2024-gs1-model-answers/',
            'https://www.civilsdaily.com/upsc-2024-gs2-model-answers/',
            'https://www.civilsdaily.com/upsc-2024-gs3-model-answers/',
        ], 10),
        ('Mrunal', ['https://mrunal.org/mains', 'https://mrunal.org/category/mains-answer-writing'], 10),
    ]

    from bs4 import BeautifulSoup
    for source_name, urls, max_articles in SOURCES:
        source_records = []
        for base_url in urls:
            html = fetch(base_url)
            if not html: continue
            soup = BeautifulSoup(html, 'html.parser')
            article_links = list(set([
                a['href'] for a in soup.select('h2 a, h3 a, .entry-title a, .post-title a')
                if a.get('href') and 'http' in a.get('href','')
            ]))[:max_articles]
            if not article_links:
                article_links = [base_url]
            for link in article_links:
                time.sleep(1.5)
                art_html = fetch(link)
                if not art_html: continue
                text = extract_body(art_html)
                year_m = re.search(r'20(\d{2})', link)
                year = int(f'20{year_m.group(1)}') if year_m else 2023
                pairs = extract_qa_pairs(text)
                for p in pairs:
                    source_records.append(TopperRecord(
                        question=p['q'], answer=p['a'],
                        subject=infer_subject(p['q']),
                        year=year, source=source_name,
                    ))
            time.sleep(2)
        all_records.extend(source_records)
        print(f'  {source_name}: {len(source_records)} records')

    print(f'\n✅ Total scraped: {len(all_records)} records')
else:
    print('✅ Skipped scraping (using pre-built dataset)')

In [ ]:
# Step 4: Build training dataset from scraped data (skip if using pre-built)
import random

ANTI_HALLUCINATION_SYSTEM = """You are a senior UPSC/HCS Mains answer evaluator with 20+ years of experience.

CRITICAL INSTRUCTION: Base your evaluation ONLY on the topper reference answer provided.
Do NOT add facts, schemes, data points, or constitutional provisions not present in the reference.

STRICT RULES:
1. keywords_found must only contain terms that literally appear in the student answer
2. keywords_missed must only reference keywords present in the topper reference
3. Scores must reflect ACTUAL content in the student answer — do not inflate
4. strengths must cite specific phrases from the student answer
5. topper_benchmark must compare against the PROVIDED reference, not general knowledge
6. Return ONLY valid JSON — no extra text before or after

Return JSON with keys:
relevance_score, intro_score, body_score, keyword_score, structure_score, factual_score,
conclusion_score, word_limit_score, analysis_score, diagram_score, multidimensional_score (0-10 each),
overall_score, marks_obtained, feedback_summary, strengths (list), weaknesses (list),
improvements (list), keywords_found (list), keywords_missed (list),
topper_benchmark (string), dimension_analysis (object: political/economic/social/environmental/ethical/legal)"""

if not USE_PREBUILT_DATASET:
    # 5-tier student answer templates
    TIERS = {
        'very_weak': {
            'answers': [
                'Good governance is very important for development.',
                'India needs more reforms. Government should work hard.',
                'This is a important topic. Many problems exist in society.',
                'Development is needed. Government should take steps.',
                'The issue is complex. More attention is needed.',
            ],
            'scores': {'relevance_score':2.5,'intro_score':1.5,'body_score':2.0,'keyword_score':1.0,
                'structure_score':2.0,'factual_score':1.0,'conclusion_score':1.0,'word_limit_score':2.5,
                'analysis_score':1.5,'diagram_score':0.0,'multidimensional_score':1.5,'overall_score':1.8},
            'feedback': 'Completely inadequate. No specific data, schemes, analysis, or Way Forward. Generic statements only.',
        },
        'weak': {
            'answers': [
                'Good governance is important. Government should be transparent. Corruption must be removed.',
                'Green Revolution helped India agriculture. But environment problems exist. Solutions needed.',
                'Federalism means power sharing between centre and states. Cooperation is needed for development.',
                'Ethics is important for civil servants. They should be honest and serve public.',
                'Climate change is serious problem. India should use renewable energy more.',
            ],
            'scores': {'relevance_score':4.5,'intro_score':3.0,'body_score':3.5,'keyword_score':2.5,
                'structure_score':3.0,'factual_score':2.5,'conclusion_score':2.0,'word_limit_score':4.0,
                'analysis_score':2.5,'diagram_score':0.0,'multidimensional_score':2.5,'overall_score':3.1},
            'feedback': 'Very basic answer with no specific data or schemes. Generic statements without evidence. No Way Forward.',
        },
        'average': {
            'answers': [
                'Good governance requires transparency and accountability. Haryana has launched SARAL platform. However challenges like corruption remain. Way Forward: more e-governance needed.',
                'Green Revolution transformed Haryana into agricultural powerhouse. But groundwater depletion and stubble burning are challenges. Sustainable practices should be promoted.',
                'India follows cooperative federalism through NITI Aayog and GST Council. However Centre-State tensions exist. ISC meetings should be regular.',
                'Civil servants face ethical challenges like political interference and corruption. Institutional safeguards like Lokpal exist but need strengthening.',
                'India committed to Paris Agreement 45% emission reduction target. Solar capacity has grown. However coal dependency creates tension.',
            ],
            'scores': {'relevance_score':6.5,'intro_score':5.5,'body_score':6.0,'keyword_score':5.5,
                'structure_score':5.5,'factual_score':5.5,'conclusion_score':5.0,'word_limit_score':7.5,
                'analysis_score':5.0,'diagram_score':0.0,'multidimensional_score':5.5,'overall_score':5.8},
            'feedback': 'Covers basics with some examples but lacks depth. Data is superficial and Way Forward is generic.',
        },
        'good': {
            'answers': [
                'Good governance embodies transparency, accountability, participation.\n\nHaryana Achievements:\n• SARAL: 550+ services, 1 crore transactions\n• CM Window: 50 lakh grievances resolved\n\nChallenges:\n1. Admin vacancy 30%; political transfers\n2. Digital divide: rural 45% vs urban 72%\n\nWay Forward:\n1. 2-year posting tenure (ARC2)\n2. District Performance Index',
                'Green Revolution Impact:\n• Wheat yield: 1.1 → 4.2 T/ha\n• 60 lakh MT wheat to Central Pool\n\nChallenges:\n1. Groundwater: 11/22 districts dark zone (CGWB)\n2. Stubble burning: 50,000 incidents\n\nSolutions:\n• Mera Pani Meri Virasat: Rs 7000/acre incentive\n• ZBNF natural farming: 50,000 acres pilot',
            ],
            'scores': {'relevance_score':8.5,'intro_score':8.0,'body_score':8.5,'keyword_score':8.0,
                'structure_score':9.0,'factual_score':8.0,'conclusion_score':8.5,'word_limit_score':9.0,
                'analysis_score':8.0,'diagram_score':0.0,'multidimensional_score':8.5,'overall_score':8.4},
            'feedback': 'Excellent answer with specific data, multi-dimensional analysis and concrete Way Forward. Near topper-level.',
        },
        'excellent': {
            'answers': [
                'Good governance (UN: ESCAP 8 pillars) requires structural reforms.\n\nHaryana Achievements:\n• SARAL: 550+ services, average delivery 3.2 days vs 30-day norm\n• CM Window: 50 lakh grievances, 94% resolution rate\n• CCTNS: 100% FIR digitization\n\nStructural Challenges:\n1. Admin: 30% vacancy; Ashok Khemka: 53 transfers in 32 years — systemic problem\n2. Digital: Rural internet 45% vs urban 72%; Nuh, Mewat worst affected\n3. Judicial: 8 lakh pending cases; 15-year avg wait for civil disputes\n4. Revenue: 30% tehsils lack complete land digitization\n\nWay Forward (ARC2-backed):\n1. Minimum 2-year posting security (ARC2 Rec 7.15) — prevent weaponized transfers\n2. District Performance Index with public citizen scorecards — accountability\n3. Gram Sachivalaya: 1 secretary per 500 households — last-mile delivery\n4. Haryana Ombudsman: 30-day mandatory resolution — teeth to oversight\n5. GIS-based revenue records + e-courts in all talukas — tech backbone\n\nGood governance is a continuous process requiring administrative, digital, and judicial reform synergy.',
            ],
            'scores': {'relevance_score':9.5,'intro_score':9.5,'body_score':9.5,'keyword_score':9.0,
                'structure_score':9.5,'factual_score':9.0,'conclusion_score':9.5,'word_limit_score':9.5,
                'analysis_score':9.0,'diagram_score':0.0,'multidimensional_score':9.5,'overall_score':9.3},
            'feedback': 'Topper-level answer. Covers all dimensions with specific data, ARC2 citations, concrete Way Forward. Minor gap: no diagram.',
        },
    }

    for record in all_records:
        if not record.answer or len(record.answer) < 100:
            continue
        for tier, tier_data in TIERS.items():
            student_answer = random.choice(tier_data['answers'])
            scores = dict(tier_data['scores'])
            scores['marks_obtained'] = round(scores['overall_score'] / 10 * 15, 1)
            scores['feedback_summary'] = tier_data['feedback']
            scores['strengths'] = ['Identified core theme'] if tier in ('very_weak','weak') else ['Specific data cited', 'Structure present', 'Multi-dimensional']
            scores['weaknesses'] = ['No specific data', 'No Way Forward', 'Generic statements']
            scores['improvements'] = ['Add schemes with data', 'Include Way Forward', 'Cover all 6 dimensions']
            scores['keywords_found'] = []
            scores['keywords_missed'] = ['transparency', 'accountability', 'ARC2']
            scores['topper_benchmark'] = f'A topper opens with definitional anchor, cites specific data, covers all 6 dimensions, and closes with ARC2-backed Way Forward.'
            scores['dimension_analysis'] = {
                'political': 'government' in student_answer.lower(),
                'economic': any(w in student_answer.lower() for w in ['economy','agriculture','industry','gdp']),
                'social': any(w in student_answer.lower() for w in ['social','education','health','gender']),
                'environmental': any(w in student_answer.lower() for w in ['environment','water','pollution','climate']),
                'ethical': any(w in student_answer.lower() for w in ['ethical','integrity','moral','corruption']),
                'legal': any(w in student_answer.lower() for w in ['law','act','article','constitution']),
            }

            user_content = f'Question ({record.subject}, 15 marks, {record.year} UPSC/HCS):\n{record.question}\n\nTopper Reference Answer (Score {record.score}/10):\n{record.answer[:2000]}\n\nStudent Answer ({len(student_answer.split())} words):\n{student_answer}'

            training_examples.append({
                'instruction': ANTI_HALLUCINATION_SYSTEM,
                'input': user_content,
                'output': json.dumps(scores, indent=2),
            })

    random.shuffle(training_examples)
    print(f'✅ Built {len(training_examples)} training examples from scraped data')

    # Save to Drive
    with open(DATASET_PATH, 'w', encoding='utf-8') as f:
        for ex in training_examples:
            f.write(json.dumps(ex, ensure_ascii=False) + '\n')
    print(f'✅ Saved dataset to Drive: {DATASET_PATH}')
else:
    print(f'✅ Using pre-built dataset: {len(training_examples)} examples')

In [ ]:
# Step 5: Load Llama-3.1-8B with Unsloth (4-bit quantized)
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit',
    max_seq_length=4096,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_alpha=16,
    lora_dropout=0.05,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)

print('✅ Model loaded with LoRA adapters')
model.print_trainable_parameters()

In [ ]:
# Step 6: Format dataset for Llama-3 chat template
from datasets import Dataset

def format_chat(example):
    messages = [
        {'role': 'system', 'content': example['instruction']},
        {'role': 'user', 'content': example['input']},
        {'role': 'assistant', 'content': example['output']},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return {'text': text}

dataset = Dataset.from_list(training_examples)
dataset = dataset.map(format_chat, remove_columns=dataset.column_names)
split = dataset.train_test_split(test_size=0.1, seed=42)

print(f'✅ Train: {len(split["train"])} | Eval: {len(split["test"])}')
print(f'   Sample (first 300 chars):')
print(split['train'][0]['text'][:300])

In [ ]:
# Step 7: FINE-TUNE — The core training step (~1-1.5 hours on T4)
from trl import SFTTrainer
from transformers import TrainingArguments
import torch

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=split['train'],
    eval_dataset=split['test'],
    dataset_text_field='text',
    max_seq_length=4096,
    packing=False,
    args=TrainingArguments(
        output_dir=MODEL_OUTPUT_PATH,
        num_train_epochs=3,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_ratio=0.1,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        eval_strategy='steps',
        eval_steps=50,
        save_steps=100,
        save_total_limit=2,
        optim='adamw_8bit',
        weight_decay=0.01,
        lr_scheduler_type='cosine',
        seed=42,
        report_to='none',
    ),
)

print(f'🚀 Starting fine-tuning...')
print(f'   Training on {len(split["train"])} examples, evaluating on {len(split["test"])}')
print(f'   Model will be saved to Google Drive: {MODEL_OUTPUT_PATH}')
stats = trainer.train()
print(f'✅ Training complete! Final loss: {stats.training_loss:.4f}')

In [ ]:
# Step 8: Save fine-tuned model to Google Drive
model.save_pretrained(MODEL_OUTPUT_PATH)
tokenizer.save_pretrained(MODEL_OUTPUT_PATH)
print(f'✅ LoRA adapters saved to Drive: {MODEL_OUTPUT_PATH}')

# Save merged 16-bit version (needed if deploying without Unsloth)
merged_path = MODEL_OUTPUT_PATH + '-merged'
model.save_pretrained_merged(merged_path, tokenizer, save_method='merged_16bit')
print(f'✅ Merged 16-bit model saved to Drive: {merged_path}')
print('\nYour model is safe in Google Drive even if Colab disconnects!')

In [ ]:
# Step 9: Test the fine-tuned model
FastLanguageModel.for_inference(model)

ANTI_HALLUCINATION_SYSTEM = """You are a senior UPSC/HCS Mains answer evaluator with 20+ years of experience.

CRITICAL INSTRUCTION: Base your evaluation ONLY on the topper reference answer provided.
Do NOT add facts, schemes, data points, or constitutional provisions not present in the reference.

STRICT RULES:
1. keywords_found must only contain terms that literally appear in the student answer
2. keywords_missed must only reference keywords present in the topper reference
3. Scores must reflect ACTUAL content in the student answer — do not inflate
4. strengths must cite specific phrases from the student answer
5. topper_benchmark must compare against the PROVIDED reference, not general knowledge
6. Return ONLY valid JSON — no extra text before or after

Return JSON with keys: relevance_score, intro_score, body_score, keyword_score, structure_score,
factual_score, conclusion_score, word_limit_score, analysis_score, diagram_score,
multidimensional_score (0-10 each), overall_score, marks_obtained, feedback_summary,
strengths (list), weaknesses (list), improvements (list), keywords_found (list),
keywords_missed (list), topper_benchmark (string),
dimension_analysis (object: political/economic/social/environmental/ethical/legal)"""

TEST_Q = 'Discuss the challenges of good governance in Haryana.'
TEST_TOPPER = 'Good governance requires transparency. SARAL platform: 550+ services, 1 crore transactions. CM Window: 50 lakh grievances resolved. Challenges: 30% admin vacancy, digital divide (rural 45% vs urban 72%). Way Forward: 2-year posting tenure (ARC2 Rec 7.15), District Performance Index, Gram Sachivalaya strengthening.'
TEST_STUDENT = 'Good governance is important. Haryana has SARAL platform. There are problems like corruption. Government should improve services and be more transparent.'

messages = [
    {'role': 'system', 'content': ANTI_HALLUCINATION_SYSTEM},
    {'role': 'user', 'content': f'Question (GS2 - Governance, 15 marks, 2025 HCS):\n{TEST_Q}\n\nTopper Reference Answer (Score 9.0/10):\n{TEST_TOPPER}\n\nStudent Answer ({len(TEST_STUDENT.split())} words):\n{TEST_STUDENT}'},
]

inputs = tokenizer.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True, return_tensors='pt'
).to('cuda')

outputs = model.generate(
    inputs, max_new_tokens=1024,
    temperature=0.1, do_sample=False,
    repetition_penalty=1.1,
)
result = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)

print('=== FINE-TUNED MODEL EVALUATION ===')
print(result)

try:
    parsed = json.loads(result)
    print(f'\n✅ Valid JSON!')
    print(f'   Overall Score: {parsed["overall_score"]}/10')
    print(f'   Marks: {parsed["marks_obtained"]}/15')
    print(f'   Feedback: {parsed["feedback_summary"]}')
    print(f'   Keywords found: {parsed["keywords_found"]}')
    print(f'   Keywords missed: {parsed["keywords_missed"]}')
except json.JSONDecodeError:
    print('⚠️  Output not valid JSON — may need more training data or epochs')

In [ ]:
# Step 10: Push to HuggingFace Hub (private) — RECOMMENDED
# This gives you a permanent URL to load the model from anywhere
#
# To get your token:
#   1. Sign up free at huggingface.co
#   2. Go to Settings → Access Tokens → New token (write permission)
#   3. Paste it below

HF_TOKEN = ''  # ← paste your HuggingFace token here
HF_REPO = ''   # ← e.g. 'your-username/hazeon-upsc-evaluator'

if HF_TOKEN and HF_REPO:
    from huggingface_hub import login
    login(token=HF_TOKEN)
    model.push_to_hub(HF_REPO, private=True)
    tokenizer.push_to_hub(HF_REPO, private=True)
    print(f'✅ Model pushed to HuggingFace: https://huggingface.co/{HF_REPO}')
    print(f'\nTo use in backend/.env add:')
    print(f'   FINETUNED_MODEL_PATH={HF_REPO}')
else:
    # Alternative: download zip from Drive
    print('HF_TOKEN not set — zipping model from Drive for download...')
    !cd {DRIVE_BASE} && zip -r /content/hazeon-upsc-evaluator.zip hazeon-upsc-evaluator/
    from google.colab import files
    files.download('/content/hazeon-upsc-evaluator.zip')
    print('\n✅ Download started!')
    print('Next steps:')
    print('  1. Extract zip to backend/models/hazeon-upsc-evaluator/')
    print('  2. Add to backend/.env:')
    print('     FINETUNED_MODEL_PATH=./models/hazeon-upsc-evaluator')